# Tier 2 — T2.1: Scaling Replication on Pythia 160M + 1.4B

Replicates the Paper 2 main pilot on two additional model sizes to test whether the four central findings (emergent > born-critical, dual differentiation, DFE shape emergence, scale hierarchy) are Pythia-410M-specific or hold across sizes.

| Size | Architecture | Heads ablated | Layer abl. | Type A | Checkpoints | Cost |
|------|--------------|---------------|-----------|--------|-------------|------|
| 160M-deduped | auto-detect | all | all layers | 30 | 8 | ~40 min A100 |
| 1.4B-deduped | 24L × 16H | 144 (6/layer, [0,3,6,9,12,15]) | all 24 | 30 | 8 | ~4 h A100 80GB |

**Runtime:** ~5 h total on A100 80GB. Can run 160M only on regular A100 / T4.

**Output:** `tier2_t21_scaling_{160m,1p4b}.csv` + `tier2_t21_summary.json` with DFE fits per checkpoint, dual-differentiation test, emergence-vs-born-critical counts.

**Resume:** every ablation appended to CSV; rerunning skips completed rows.

In [ ]:
!pip install -q transformers datasets torch accelerate scipy pandas

In [ ]:
import torch, json, os, time, csv, hashlib, gc, urllib.request
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU: {torch.cuda.get_device_name()}  |  Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT_DIR = '/content/drive/MyDrive/DFE_research/tier2'
    os.makedirs(OUT_DIR, exist_ok=True)
    test = os.path.join(OUT_DIR, '.w')
    with open(test, 'w') as f: f.write('ok')
    os.remove(test)
    print(f'Drive mounted and verified; output to {OUT_DIR}')
except Exception as e:
    raise RuntimeError(f'Drive mount required: {e}')

GITHUB_RAW = 'https://raw.githubusercontent.com/mool32/functional-differentiation-dfe/main'

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

## Shared config + helpers

In [ ]:
CHECKPOINTS = [512, 1000, 2000, 4000, 8000, 16000, 64000, 143000]
EVAL_N_BATCHES = 25
EVAL_BATCH_SIZE = 4
EVAL_SEQ_LEN = 2048
TYPE_A_ALPHA = 1.0
N_TYPE_A = 30
SEED_BASE = 42000  # same as Paper 2

CSV_FIELDS = ['checkpoint', 'perturbation_type', 'subtype', 'layer_idx', 'head_idx',
              'seed', 'baseline_loss', 'perturbed_loss', 'delta', 'elapsed_sec']

def csv_append(path, row):
    new = not os.path.exists(path)
    with open(path, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if new: w.writeheader()
        w.writerow(row)

def load_completed(path):
    if not os.path.exists(path): return set()
    df = pd.read_csv(path)
    return set(zip(df['checkpoint'], df['perturbation_type'], df['subtype'],
                   df['layer_idx'].fillna(-1).astype(int),
                   df['head_idx'].fillna(-1).astype(int),
                   df['seed'].fillna(-1).astype(int)))

def tensor_hash(t):
    return hashlib.sha256(t.detach().cpu().contiguous().numpy().tobytes()).hexdigest()[:16]

## Ablation + restore primitives

In [ ]:
def ablate_head(model, L, H):
    hd = model.config.hidden_size // model.config.num_attention_heads
    w = model.gpt_neox.layers[L].attention.dense.weight
    saved = w.data.clone()
    w.data[:, H*hd:(H+1)*hd] = 0
    return saved

def restore_head(model, L, saved):
    model.gpt_neox.layers[L].attention.dense.weight.data.copy_(saved)

def ablate_layer(model, L):
    block = model.gpt_neox.layers[L]
    saved = {n: p.data.clone() for n, p in block.named_parameters()}
    for _, p in block.named_parameters():
        p.data.zero_()
    return saved

def restore_block(model, L, saved):
    block = model.gpt_neox.layers[L]
    for n, p in block.named_parameters():
        p.data.copy_(saved[n])

def perturb_type_a(model, L, alpha, seed, device):
    g = torch.Generator(device=device).manual_seed(seed)
    block = model.gpt_neox.layers[L]
    saved = {n: p.data.clone() for n, p in block.named_parameters()}
    for _, p in block.named_parameters():
        noise = torch.randn(p.shape, generator=g, device=device, dtype=p.dtype) * (p.data.std() * alpha)
        p.data.add_(noise)
    return saved

@torch.no_grad()
def evaluate_loss(model, batches, device):
    total = 0.0
    for i in range(batches.shape[0]):
        ids = batches[i].to(device)
        total += model(input_ids=ids, labels=ids).loss.item()
    return total / batches.shape[0]

## Prepare eval batches once (wikitext-103)

In [ ]:
from datasets import load_dataset

def stream_batches(tok, spec, n_batches=EVAL_N_BATCHES, batch_size=EVAL_BATCH_SIZE, seq_len=EVAL_SEQ_LEN):
    ds = load_dataset(*spec[:-1], split=spec[-1], streaming=True)
    need = n_batches * batch_size * seq_len
    toks = []
    for ex in ds:
        txt = ex.get('text', ex.get('content', ''))
        if not txt or len(txt.strip()) < 50: continue
        ids = tok(txt, return_tensors='pt', truncation=False)['input_ids'].squeeze()
        if ids.dim() == 0: continue
        toks.append(ids)
        if sum(t.numel() for t in toks) >= need * 1.2: break
    merged = torch.cat(toks)[:need]
    return merged.reshape(n_batches, batch_size, seq_len)

## Main sweep loop — reusable across sizes

In [ ]:
def run_sweep(model_name, head_sample, layer_sample, csv_path, device='cuda'):
    tok = AutoTokenizer.from_pretrained(model_name)
    log(f'Tokenizer loaded: {model_name}')

    batches_path = os.path.join(OUT_DIR, f'batches_{model_name.split("/")[-1]}.pt')
    if os.path.exists(batches_path):
        batches = torch.load(batches_path, weights_only=True)
        log(f'Loaded cached batches: {batches.shape}')
    else:
        batches = stream_batches(tok, ('wikitext', 'wikitext-103-raw-v1', 'train'))
        torch.save(batches, batches_path)
        log(f'Cached batches: {batches.shape}')

    completed = load_completed(csv_path)
    log(f'CSV has {len(completed)} completed rows; will skip.')

    for step in CHECKPOINTS:
        log(f'=== CHECKPOINT step{step} ===')
        t_ckpt = time.time()
        model = AutoModelForCausalLM.from_pretrained(
            model_name, revision=f'step{step}', torch_dtype=torch.float32
        ).to(device).eval()

        bl = evaluate_loss(model, batches, device)
        log(f'  baseline loss: {bl:.4f}')

        # Head ablations
        for (L, H) in head_sample:
            key = (step, 'head', 'output_zeroing', L, H, -1)
            if key in completed: continue
            t0 = time.time()
            saved = ablate_head(model, L, H)
            pl = evaluate_loss(model, batches, device)
            restore_head(model, L, saved)
            csv_append(csv_path, {
                'checkpoint': step, 'perturbation_type': 'head', 'subtype': 'output_zeroing',
                'layer_idx': L, 'head_idx': H, 'seed': -1,
                'baseline_loss': bl, 'perturbed_loss': pl,
                'delta': -(pl - bl) / abs(bl), 'elapsed_sec': time.time() - t0,
            })

        # Layer ablations
        for L in layer_sample:
            key = (step, 'layer', 'full_zero', L, -1, -1)
            if key in completed: continue
            t0 = time.time()
            saved = ablate_layer(model, L)
            pl = evaluate_loss(model, batches, device)
            restore_block(model, L, saved)
            csv_append(csv_path, {
                'checkpoint': step, 'perturbation_type': 'layer', 'subtype': 'full_zero',
                'layer_idx': L, 'head_idx': -1, 'seed': -1,
                'baseline_loss': bl, 'perturbed_loss': pl,
                'delta': -(pl - bl) / abs(bl), 'elapsed_sec': time.time() - t0,
            })

        # Type A
        n_layers = len(model.gpt_neox.layers)
        for i in range(N_TYPE_A):
            L = i % n_layers
            seed = SEED_BASE + i
            key = (step, 'type_a', 'block_noise', L, -1, seed)
            if key in completed: continue
            t0 = time.time()
            saved = perturb_type_a(model, L, TYPE_A_ALPHA, seed, device)
            pl = evaluate_loss(model, batches, device)
            restore_block(model, L, saved)
            csv_append(csv_path, {
                'checkpoint': step, 'perturbation_type': 'type_a', 'subtype': 'block_noise',
                'layer_idx': L, 'head_idx': -1, 'seed': seed,
                'baseline_loss': bl, 'perturbed_loss': pl,
                'delta': -(pl - bl) / abs(bl), 'elapsed_sec': time.time() - t0,
            })

        log(f'  checkpoint done in {(time.time() - t_ckpt)/60:.1f} min')
        del model; gc.collect(); torch.cuda.empty_cache()

    log('sweep complete')

## Run Pythia 160M-deduped

Auto-detects layer/head counts from config and ablates all heads + all layers.

In [ ]:
MODEL_160M = 'EleutherAI/pythia-160m-deduped'
CSV_160M = os.path.join(OUT_DIR, 'tier2_t21_scaling_160m.csv')

_tmp = AutoModelForCausalLM.from_pretrained(MODEL_160M, revision='step143000', torch_dtype=torch.float32)
N_L_160M = len(_tmp.gpt_neox.layers)
N_H_160M = _tmp.config.num_attention_heads
del _tmp; gc.collect()
log(f'160M architecture: {N_L_160M} layers x {N_H_160M} heads = {N_L_160M*N_H_160M} heads total')

head_sample_160m = [(L, H) for L in range(N_L_160M) for H in range(N_H_160M)]
layer_sample_160m = list(range(N_L_160M))
log(f'160M total ablations per checkpoint: {len(head_sample_160m) + len(layer_sample_160m) + N_TYPE_A}')

run_sweep(MODEL_160M, head_sample_160m, layer_sample_160m, CSV_160M)

## Run Pythia 1.4B-deduped (optional — needs A100 80GB)

Samples 144 heads ([0,3,6,9,12,15] × 24 layers) matching Paper 2's 410M sampling. Skip this cell on smaller GPUs.

In [ ]:
MODEL_14B = 'EleutherAI/pythia-1.4b-deduped'
CSV_14B = os.path.join(OUT_DIR, 'tier2_t21_scaling_1p4b.csv')

FIXED_HEADS_14B = [0, 3, 6, 9, 12, 15]  # 6 per layer -> 144 total
N_L_14B = 24  # Pythia 1.4B
N_H_14B = 16

head_sample_14b = [(L, H) for L in range(N_L_14B) for H in FIXED_HEADS_14B]
layer_sample_14b = list(range(N_L_14B))
log(f'1.4B total ablations per checkpoint: {len(head_sample_14b) + len(layer_sample_14b) + N_TYPE_A}')

# Guard: require A100 80GB
if device == 'cuda':
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gb < 60:
        print(f'WARNING: GPU has {gb:.0f} GB; 1.4B needs ~60 GB for float32 forward. Skipping.')
    else:
        run_sweep(MODEL_14B, head_sample_14b, layer_sample_14b, CSV_14B)

## Analysis — DFE fits + dual differentiation + emergence counts

In [ ]:
def analyze(csv_path, label):
    if not os.path.exists(csv_path):
        log(f'{label}: CSV not found, skipping')
        return None
    df = pd.read_csv(csv_path)
    log(f'{label}: {len(df)} rows, {df["checkpoint"].nunique()} checkpoints')
    results = {'label': label, 'per_checkpoint': {}, 'findings': {}}

    for step in sorted(df['checkpoint'].unique()):
        d = df[df['checkpoint'] == step]
        heads = d[d['perturbation_type'] == 'head']['delta'].values
        type_a = d[d['perturbation_type'] == 'type_a']['delta'].values

        # Threshold-free: Gini of |negative deltas| + effective N
        neg = -heads[heads < -1e-4]
        if len(neg) >= 5:
            sorted_neg = np.sort(neg)
            n = len(sorted_neg)
            gini = (2 * np.sum((np.arange(1, n+1)) * sorted_neg)) / (n * sorted_neg.sum()) - (n + 1) / n
            eff_n = sorted_neg.sum() ** 2 / (sorted_neg ** 2).sum()
            count_neg = len(neg)
            # beta fit (Gamma shape)
            try:
                b, _, _ = sp_stats.gamma.fit(neg, floc=0)
            except Exception:
                b = float('nan')
        else:
            gini = eff_n = count_neg = b = float('nan')

        # Type A Student's t vs normal
        if len(type_a) >= 10:
            df_t, loc_t, sc_t = sp_stats.t.fit(type_a)
            aic_t = 2*3 - 2*np.sum(sp_stats.t.logpdf(type_a, df_t, loc_t, sc_t))
            mu, si = sp_stats.norm.fit(type_a)
            aic_n = 2*2 - 2*np.sum(sp_stats.norm.logpdf(type_a, mu, si))
            d_aic = float(aic_n - aic_t)
        else:
            df_t = d_aic = float('nan')

        results['per_checkpoint'][int(step)] = {
            'n_heads': len(heads), 'n_type_a': len(type_a),
            'heads_gini': float(gini), 'heads_eff_n': float(eff_n),
            'heads_count_neg': int(count_neg) if not np.isnan(count_neg) else -1,
            'heads_beta': float(b),
            'type_a_student_t_df': float(df_t), 'type_a_delta_aic_t_vs_n': d_aic,
        }

    # Dual differentiation test: from first to last checkpoint
    first_step = min(df['checkpoint'])
    last_step = max(df['checkpoint'])
    f = results['per_checkpoint'][int(first_step)]
    l = results['per_checkpoint'][int(last_step)]
    results['findings']['dual_differentiation'] = {
        'count_neg_first': f['heads_count_neg'], 'count_neg_last': l['heads_count_neg'],
        'eff_n_first': f['heads_eff_n'], 'eff_n_last': l['heads_eff_n'],
        'gini_first': f['heads_gini'], 'gini_last': l['heads_gini'],
        'signature_present': (l['heads_count_neg'] > f['heads_count_neg']
                              and l['heads_eff_n'] < f['heads_eff_n']
                              and abs(l['heads_gini'] - f['heads_gini']) < 0.1),
    }

    # Emergence: head critical at final but not at first
    crit_thresh = 5e-4
    crit_final = df[(df['checkpoint'] == last_step) & (df['perturbation_type'] == 'head')
                    & (df['delta'].abs() > crit_thresh)]
    crit_initial = df[(df['checkpoint'] == first_step) & (df['perturbation_type'] == 'head')
                      & (df['delta'].abs() > crit_thresh)]
    final_keys = set(zip(crit_final['layer_idx'], crit_final['head_idx']))
    initial_keys = set(zip(crit_initial['layer_idx'], crit_initial['head_idx']))
    emergent = final_keys - initial_keys
    born = final_keys & initial_keys
    results['findings']['emergence'] = {
        'n_born_critical': len(born),
        'n_emergent': len(emergent),
        'ratio': (len(emergent) / max(len(born), 1)),
    }

    return results

res_160m = analyze(os.path.join(OUT_DIR, 'tier2_t21_scaling_160m.csv'), '160M')
res_14b = analyze(os.path.join(OUT_DIR, 'tier2_t21_scaling_1p4b.csv'), '1.4B')

summary = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    '160m': res_160m,
    '1p4b': res_14b,
}

summary_path = os.path.join(OUT_DIR, 'tier2_t21_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print('\n=== T2.1 SUMMARY ===')
for key in ('160m', '1p4b'):
    r = summary[key]
    if r is None: continue
    print(f'\n[{r["label"]}]')
    dd = r['findings']['dual_differentiation']
    print(f'  dual_differentiation signature present: {dd["signature_present"]}')
    print(f'    count_neg: {dd["count_neg_first"]} -> {dd["count_neg_last"]}')
    print(f'    eff_n: {dd["eff_n_first"]:.2f} -> {dd["eff_n_last"]:.2f}')
    print(f'    gini: {dd["gini_first"]:.3f} -> {dd["gini_last"]:.3f}')
    em = r['findings']['emergence']
    print(f'  emergence: born={em["n_born_critical"]}  emergent={em["n_emergent"]}  ratio={em["ratio"]:.2f}')

log(f'saved to {summary_path}')

try:
    from google.colab import files
    for p in [os.path.join(OUT_DIR, 'tier2_t21_scaling_160m.csv'),
              os.path.join(OUT_DIR, 'tier2_t21_scaling_1p4b.csv'),
              summary_path]:
        if os.path.exists(p):
            files.download(p)
except Exception:
    pass